# Graph RAG — COSORA (experimental)

Pipeline COSORA + Knowledge Graph generado con **kg-gen** sobre los chunks ya indexados en ChromaDB.

**Idea:** dense (E5) + BM25 + **triples del grafo** fusionados con RRF.
Genera el grafo una sola vez (offline) con `gpt-4o-mini`, lo persiste en JSON
y opcionalmente lo sube a GCS junto al Chroma.

**Prerequisito:** haber ejecutado `notebooks/ingestion_pipeline.ipynb` con `EMBED_BACKEND="e5"`.

> Autocontenido. Funciona en local (kernel `.venv`) y en Colab.


## 0. Setup
Instalación de dependencias, auto-detección Colab/local y código COSORA embebido.
Incluye `kg-gen` (PyPI, no `git clone`).


In [1]:
%pip install -q chromadb sentence-transformers rank_bm25 transformers accelerate python-dotenv python-docx nltk pandas torch kg-gen google-cloud-storage

# antiword (para .doc legacy) — sólo lo necesita KG_SOURCE="original"
import shutil, subprocess
if shutil.which("antiword") is None:
    try:
        subprocess.run(["apt-get", "install", "-y", "-q", "antiword"], check=False)
    except Exception:
        print("⚠️  antiword no se pudo instalar. KG_SOURCE='original' sólo leerá .docx.")

import nltk
nltk.download('punkt', quiet=True)


True

In [2]:
# ═══════════════════════════════════════════════════════════════════════
# AUTO-DETECT: Colab vs Local  +  Configuración
# ═══════════════════════════════════════════════════════════════════════
import glob
import json
import os
import re
import subprocess
import sys
import time
import unicodedata
from pathlib import Path
from typing import Any

import chromadb
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from transformers import AutoModel, AutoTokenizer

IN_COLAB = "google.colab" in sys.modules

# ─── Configuración ────────────────────────────────────────────────────
RUNTIME = "auto"            # "auto" | "colab" | "local"
EMBED_BACKEND = "e5"        # fijado a e5 para alinear con producción (src/app.py)

GEN_BACKEND = "openai"

RETRIEVAL_K = 100
TOP_N = 10
RRF_K = 60
RRF_MIN_SCORE = 0.015
SCORE_RATIO = 0.60

# ─── Graph RAG ────────────────────────────────────────────────────────
KG_MODEL = "openai/gpt-4o-mini"   # litellm-style, lo usa kg-gen
KG_CONTEXT = "Actas de obra ferroviaria en España. Personas, empresas (UTE, DF, DEO), elementos constructivos, incidencias, fechas."
KG_SOURCES = ["chunks", "hybrid", "original"]   # se generan y comparan los 3
                                  #   chunks   = N chunks tal cual salen de Chroma (más granular, menos contexto)
                                  #   hybrid   = chunks AGRUPADOS por doc_id → un texto por acta (sin tocar .doc/.docx)
                                  #   original = lee .doc/.docx desde RAW_DOCS_PATH (replica ingest.py)
KG_SOURCE_PRIMARY = "hybrid"      # cuál se usa para las pruebas interactivas (secciones 3-7)
SUBSET_SIZE = 10                  # nº de elementos para pruebas (chunks o actas según fuente). None = todos
KG_CHUNK_SIZE = 5000              # tamaño de chunk interno de kg-gen (chars)
KG_CLUSTER = True                 # entity resolution
GRAPH_TOP_K = 5                   # nº de triples a recuperar por query
GRAPH_MIN_COS = 0.72              # umbral mínimo de coseno para que un triple boostee chunks
                                  # (evita ruido cuando ningún triple del grafo es realmente relevante)
RAW_DOCS_PATH_OVERRIDE = None     # None = autodetectado (data/raw o Drive). Solo si KG_SOURCE="original"

# Persistencia
GRAPH_JSON_NAME = "graph_actas_e5.json"
GCS_BUCKET = os.getenv("GCP_BUCKET_NAME", "")
GCS_GRAPH_PREFIX = "graph"

if RUNTIME == "auto":
    RUNTIME = "colab" if IN_COLAB else "local"

if RUNTIME == "colab" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# ─── .env ─────────────────────────────────────────────────────────────
from dotenv import load_dotenv
if RUNTIME == "colab":
    load_dotenv("/content/drive/MyDrive/variablentorno/.env")
elif Path(".env").exists():
    load_dotenv(".env")
elif Path("../.env").exists():
    load_dotenv("../.env")
elif Path("../../.env").exists():
    load_dotenv("../../.env")


# ─── Rutas ────────────────────────────────────────────────────────────
def resolve_paths(runtime, chroma_path_override=None):
    if runtime == "colab":
        docs_dir = "/content/drive/MyDrive/RAG_UPC_Final_project"
        chroma_path = chroma_path_override or f"{docs_dir}/chroma_db"
        graph_dir = f"{docs_dir}/graph"
    else:
        nb_dir = Path(".").resolve()
        if nb_dir.name in ("experiments", "notebooks"):
            root = nb_dir.parent if nb_dir.name == "experiments" else nb_dir.parent
            root = nb_dir.parents[1] if nb_dir.name == "experiments" else nb_dir.parent
        else:
            root = nb_dir
        docs_dir = str(root / "data" / "raw")
        chroma_path = chroma_path_override or str(root / "data" / "chroma_db")
        graph_dir = str(root / "data" / "graph")
    Path(graph_dir).mkdir(parents=True, exist_ok=True)
    return docs_dir, chroma_path, graph_dir

# ═══════════════════════════════════════════════════════════════════════
# COSORA SHARED — código embebido
# ═══════════════════════════════════════════════════════════════════════

EMBED_BACKENDS: dict[str, dict[str, Any]] = {
    "mrbert": {
        "model_id": "BSC-LT/MrBERT-es",
        "collection": "cosora_chunks_mrbert",
        "type": "transformers_cls",
        "query_prefix": "",
        "doc_prefix": "",
    },
    "e5": {
        "model_id": "intfloat/multilingual-e5-base",
        "collection": "cosora_chunks_e5",
        "type": "sentence_transformers",
        "query_prefix": "query: ",
        "doc_prefix": "passage: ",
    },
}

TECH_TOKEN_PATTERN = re.compile(
    r"^(AR-\d+|UTE|DF|DEO|DO|PPI|[A-Z]{2,}\d*|\d+[A-Z-]+)$",
    re.IGNORECASE,
)

STOPWORDS_ES = {
    "de","la","que","el","en","y","a","los","del","se","las","por","un",
    "para","con","no","una","su","al","es","lo","como","más","pero","sus",
    "le","ya","o","este","sí","porque","esta","entre","cuando","muy","sin",
    "sobre","también","me","hasta","hay","donde","quien","desde","todo","nos",
    "durante","todos","uno","les","ni","contra","otros","ese","eso","ante",
}


def backend_config(backend):
    if backend not in EMBED_BACKENDS:
        raise ValueError(f"EMBED_BACKEND desconocido: {backend}")
    return EMBED_BACKENDS[backend]


class Embedder:
    def __init__(self, backend):
        self.backend = backend
        self.cfg = backend_config(backend)
        self._st_model = None
        self._model = None
        self._tokenizer = None

    def _load(self):
        if self.cfg["type"] == "sentence_transformers":
            if self._st_model is None:
                from sentence_transformers import SentenceTransformer
                self._st_model = SentenceTransformer(self.cfg["model_id"])
            return
        if self._model is None:
            self._tokenizer = AutoTokenizer.from_pretrained(self.cfg["model_id"])
            self._model = AutoModel.from_pretrained(self.cfg["model_id"])
            self._model.eval()

    def embed_one(self, text, *, is_query=False):
        self._load()
        if self.cfg["type"] == "sentence_transformers":
            prefix = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
            vec = self._st_model.encode(prefix + text)
            return vec.tolist() if hasattr(vec, "tolist") else list(vec)
        assert self._tokenizer is not None and self._model is not None
        inputs = self._tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = self._model(**inputs)
        return outputs.last_hidden_state[:, 0, :].squeeze().tolist()

    def embed_batch(self, texts, *, is_query=False, batch_size=16):
        self._load()
        if self.cfg["type"] == "sentence_transformers":
            prefix = self.cfg["query_prefix"] if is_query else self.cfg["doc_prefix"]
            prefixed = [prefix + t for t in texts]
            vecs = self._st_model.encode(prefixed, batch_size=batch_size, show_progress_bar=True)
            return vecs.tolist()
        out = []
        for t in texts:
            out.append(self.embed_one(t, is_query=is_query))
        return out


def strip_doc_prefix(text, backend):
    prefix = backend_config(backend).get("doc_prefix", "")
    if prefix and text.startswith(prefix):
        return text[len(prefix):]
    return text


def get_chroma_collection(chroma_path, backend):
    cfg = backend_config(backend)
    client = chromadb.PersistentClient(path=chroma_path)
    return client.get_collection(cfg["collection"]), cfg


def tokenize_bm25(text, stemmer=None):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    words = [w for w in text.split() if w and w not in STOPWORDS_ES]
    if stemmer is None:
        return words
    out = []
    for w in words:
        if TECH_TOKEN_PATTERN.match(w):
            out.append(w)
        else:
            out.append(stemmer.stem(w))
    return out


def build_bm25_index(collection):
    try:
        from nltk.stem.snowball import SnowballStemmer
        stemmer = SnowballStemmer("spanish")
    except Exception:
        stemmer = None
    data = collection.get()
    docs = data["documents"]
    metas = data["metadatas"]
    tokenized = [tokenize_bm25(d, stemmer) for d in docs]
    return BM25Okapi(tokenized), docs, metas, stemmer


def dense_search(collection, embedder, query, k=100):
    q_vec = embedder.embed_one(query, is_query=True)
    res = collection.query(query_embeddings=[q_vec], n_results=k)
    return [
        {"text": doc, "meta": meta, "rank_dense": i}
        for i, (doc, meta) in enumerate(zip(res["documents"][0], res["metadatas"][0]))
    ]


def bm25_search(query, bm25_index, docs, metas, stemmer, k=100):
    scores = bm25_index.get_scores(tokenize_bm25(query, stemmer))
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [
        {"text": docs[i], "meta": metas[i], "rank_bm25": rank}
        for rank, i in enumerate(top_idx)
    ]


def rrf_fusion_baseline(dense, bm25, rrf_k=60, top_n=10):
    """RRF clásico: dense + bm25 (baseline sin grafo)."""
    scores = {}
    for item in dense:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_dense"])
    for item in bm25:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_bm25"])
    return sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_n]


# ─── Inicializar ──────────────────────────────────────────────────────
DOCS_DIR, CHROMA_PATH, GRAPH_DIR = resolve_paths(RUNTIME)
collection, cfg = get_chroma_collection(CHROMA_PATH, EMBED_BACKEND)
embedder = Embedder(EMBED_BACKEND)
bm25_index, all_docs, all_metas, stemmer = build_bm25_index(collection)

print(f"IN_COLAB={IN_COLAB}  RUNTIME={RUNTIME}")
print(f"EMBED_BACKEND={EMBED_BACKEND}")
print(f"Chroma: {cfg['collection']} ({collection.count()} chunks) @ {CHROMA_PATH}")
print(f"Graph dir: {GRAPH_DIR}")
print("\n✅ Setup COSORA listo")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True  RUNTIME=colab
EMBED_BACKEND=e5
Chroma: cosora_chunks_e5 (582 chunks) @ /content/drive/MyDrive/RAG_UPC_Final_project/chroma_db
Graph dir: /content/drive/MyDrive/RAG_UPC_Final_project/graph

✅ Setup COSORA listo


## 1. Retrieval híbrido baseline (dense + BM25)
Idéntico al de `src/app.py` y `query_pipeline.ipynb`. Lo usaremos como **baseline** contra el cual comparar Graph RAG.


In [3]:
def retrieve_baseline(query, top_n=TOP_N):
    d = dense_search(collection, embedder, query, k=RETRIEVAL_K)
    b = bm25_search(query, bm25_index, all_docs, all_metas, stemmer, k=RETRIEVAL_K)
    return rrf_fusion_baseline(d, b, rrf_k=RRF_K, top_n=top_n)


# Sanity check
preview = retrieve_baseline("¿Qué se decidió sobre el talud?", top_n=3)
for h in preview:
    print(f"[{h['meta']['doc_id']}] score={h['score']:.4f} | {strip_doc_prefix(h['text'], EMBED_BACKEND)[:90]}...")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[254275-DO-AVO-15-V07] score=0.0333 | adas en los informes de revisión del PC y PGMA. AR-29 Estabilidad del talud || Posterior a...
[254275-DO-AVO-14-V07] score=0.0325 | envía en resultado de la verificación de la documentación aportada por la UTE sobre la est...
[254275-DO-AVO-16-V07] score=0.0323 | iará el certificado a la DF. AR-21 Plano camino provisional || DF solicita a la UTE que ap...


## 2. Construcción del grafo con `kg-gen`

Generamos triples `(sujeto, relación, objeto)` desde los chunks de Chroma usando `gpt-4o-mini` vía `kg-gen`.

**Estrategia para no quemar API:**
- `SUBSET_SIZE` controla cuántos chunks procesamos (configurado arriba en Setup).
- `cluster=True` ejecuta entity resolution (UTE = contratista = constructora, etc.).
- El grafo se persiste en JSON; si ya existe, se carga en vez de regenerarse.

**Coste estimado:** ~0,03 USD por los ~400 chunks completos con `gpt-4o-mini`.


In [4]:
# Preparar textos para kg-gen según KG_SOURCE
from collections import defaultdict

def _prepare_from_chunks():
    data = collection.get(include=["documents", "metadatas"])
    texts = [strip_doc_prefix(d, EMBED_BACKEND) for d in data["documents"]]
    sources = [m["chunk_id"] for m in data["metadatas"]]
    return texts, sources, "chunks de Chroma (tal cual)"


def _prepare_hybrid():
    data = collection.get(include=["documents", "metadatas"])
    by_doc = defaultdict(list)
    for d, m in zip(data["documents"], data["metadatas"]):
        by_doc[m["doc_id"]].append(strip_doc_prefix(d, EMBED_BACKEND))
    doc_ids = sorted(by_doc.keys())
    texts = ["\n\n".join(by_doc[d]) for d in doc_ids]
    return texts, doc_ids, "actas reconstruidas desde Chroma (chunks agrupados por doc_id)"


def _prepare_original():
    raw_dir = RAW_DOCS_PATH_OVERRIDE
    if raw_dir is None:
        raw_dir = DOCS_DIR if Path(DOCS_DIR).exists() else None
        if raw_dir is None:
            candidates = [
                Path("../data/raw"),
                Path("../../data/raw"),
                Path("data/raw"),
            ]
            raw_dir = next((str(c.resolve()) for c in candidates if c.exists()), None)
    if raw_dir is None or not Path(raw_dir).exists():
        raise FileNotFoundError(
            f"No se encontró carpeta de actas originales. Define RAW_DOCS_PATH_OVERRIDE."
        )

    # Loaders idénticos a src/ingest.py
    from docx import Document
    import subprocess as _sp

    def _normalize(t):
        return re.sub(r"\s+", " ", t).strip()

    def _extract_docx(fp):
        try:
            doc = Document(fp)
        except Exception as e:
            print(f"  ⚠️ {fp.name}: {e}")
            return None
        parts = []
        for p in doc.paragraphs:
            t = _normalize(p.text)
            if t:
                parts.append(t)
        for table in doc.tables:
            for row in table.rows:
                row_texts, seen = [], set()
                for cell in row.cells:
                    t = _normalize(cell.text)
                    if t and t not in seen:
                        seen.add(t)
                        row_texts.append(t)
                if row_texts:
                    parts.append(" || ".join(row_texts))
        return "\n".join(parts).strip() or None

    def _extract_doc(fp):
        try:
            r = _sp.run(["antiword", str(fp)], capture_output=True, text=True)
            return r.stdout.strip() if r.returncode == 0 else None
        except Exception as e:
            print(f"  ⚠️ antiword falló para {fp.name}: {e} (instala antiword o usa KG_SOURCE='hybrid')")
            return None

    files = sorted(list(Path(raw_dir).glob("*.docx")) + list(Path(raw_dir).glob("*.doc")))
    if not files:
        raise FileNotFoundError(f"Sin .doc/.docx en {raw_dir}")

    texts, sources = [], []
    for fp in files:
        raw = _extract_docx(fp) if fp.suffix.lower() == ".docx" else _extract_doc(fp)
        if raw:
            texts.append(raw)
            sources.append(fp.stem)
    return texts, sources, f"originales .doc/.docx desde {raw_dir}"


_PREPS = {"chunks": _prepare_from_chunks, "hybrid": _prepare_hybrid, "original": _prepare_original}


def prepare_inputs(src: str, subset_size=SUBSET_SIZE):
    """Devuelve (textos, fuentes, descripcion) según el modo."""
    if src not in _PREPS:
        raise ValueError(f"KG_SOURCE inválido: {src}. Usa: {list(_PREPS)}")
    texts, sources, desc = _PREPS[src]()
    if subset_size is not None and subset_size < len(texts):
        texts = texts[:subset_size]
        sources = sources[:subset_size]
    return texts, sources, desc


for _src in KG_SOURCES:
    try:
        _t, _s, _d = prepare_inputs(_src)
        print(f"[{_src:>8}] {len(_t)} elementos | {sum(len(t) for t in _t):,} chars | {_d}")
    except Exception as e:
        print(f"[{_src:>8}] ⚠️  No disponible: {e}")


[  chunks] 10 elementos | 4,329 chars | chunks de Chroma (tal cual)


[  hybrid] 10 elementos | 21,965 chars | actas reconstruidas desde Chroma (chunks agrupados por doc_id)
[original] 10 elementos | 163,579 chars | originales .doc/.docx desde /content/drive/MyDrive/RAG_UPC_Final_project


In [5]:
from kg_gen import KGGen

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY no encontrada — revisa tu .env")

kg = KGGen(model=KG_MODEL, temperature=0.0, api_key=api_key)
print(f"✅ KGGen inicializado con {KG_MODEL}")


✅ KGGen inicializado con openai/gpt-4o-mini


In [ ]:
_subset_tag = f"sub{SUBSET_SIZE}" if SUBSET_SIZE else "full"


def graph_json_path(src: str) -> Path:
    return Path(GRAPH_DIR) / f"graph_actas_e5_{src}_{_subset_tag}.json"


def graph_to_dict(graph, src: str, texts, sources):
    return {
        "entities": sorted(list(graph.entities)),
        "edges": sorted(list(graph.edges)),
        "relations": [list(r) for r in graph.relations],
        "entity_clusters": {k: sorted(list(v)) for k, v in (graph.entity_clusters or {}).items()},
        "edge_clusters": {k: sorted(list(v)) for k, v in (graph.edge_clusters or {}).items()},
        "meta": {
            "model": KG_MODEL,
            "kg_source": src,
            "subset_size": SUBSET_SIZE,
            "n_inputs": len(texts),
            "sources": sources,
            "context": KG_CONTEXT,
            "cluster": KG_CLUSTER,
        },
    }


def build_and_persist(src: str):
    """Genera el grafo elemento a elemento (con progress) y los agrega al final.

    En vez de pasar todo el corpus a kg.generate (que tarda 10+ min sin output),
    procesamos cada elemento por separado e imprimimos progreso. Luego kg.aggregate
    combina los grafos y kg.cluster hace entity resolution global UNA sola vez.
    """
    out = graph_json_path(src)
    if out.exists():
        print(f"[{src:>8}] 📁 ya existe → {out.name} (no se regenera)")
        with open(out, "r", encoding="utf-8") as f:
            return json.load(f)
    texts, sources, desc = prepare_inputs(src)
    if not texts:
        print(f"[{src:>8}] ⚠️  Sin textos, salto.")
        return None
    total_chars = sum(len(t) for t in texts)
    print(f"[{src:>8}] {len(texts)} elementos ({total_chars:,} chars) — generando elemento a elemento...")
    t0 = time.perf_counter()
    partial_graphs = []
    for i, txt in enumerate(texts, 1):
        ti = time.perf_counter()
        # cluster=False aquí: clusterizamos UNA sola vez al final, más barato y mejor
        g_i = kg.generate(input_data=txt, context=KG_CONTEXT, chunk_size=KG_CHUNK_SIZE, cluster=False)
        partial_graphs.append(g_i)
        elapsed = time.perf_counter() - t0
        eta = elapsed / i * (len(texts) - i)
        print(
            f"[{src:>8}] {i:>3}/{len(texts)}  {sources[i-1][:40]:<40}  "
            f"+{len(g_i.relations):>4} triples  ({time.perf_counter()-ti:.1f}s)  "
            f"ETA {eta/60:.1f} min"
        )
    print(f"[{src:>8}] Agregando {len(partial_graphs)} grafos parciales...")
    graph = kg.aggregate(partial_graphs)
    if KG_CLUSTER and graph.relations:
        print(f"[{src:>8}] Clustering global (entity resolution)...")
        graph = kg.cluster(graph, context=KG_CONTEXT)
    elapsed = time.perf_counter() - t0
    print(f"[{src:>8}] ✅ {len(graph.relations)} triples totales en {elapsed/60:.1f} min")
    gd = graph_to_dict(graph, src, texts, sources)
    with open(out, "w", encoding="utf-8") as f:
        json.dump(gd, f, ensure_ascii=False, indent=2)
    print(f"[{src:>8}] 💾 {out}")
    return gd


graphs_by_source = {}
for _src in KG_SOURCES:
    try:
        gd = build_and_persist(_src)
        if gd is not None:
            graphs_by_source[_src] = gd
    except Exception as e:
        print(f"[{_src:>8}] ❌ Error: {e}")

print(f"\nGrafos disponibles: {list(graphs_by_source)}")

# Carga el grafo primario en variables globales para las secciones interactivas 3-7
if KG_SOURCE_PRIMARY in graphs_by_source:
    graph_dict = graphs_by_source[KG_SOURCE_PRIMARY]
    print(f"Grafo primario para secciones 3-7: {KG_SOURCE_PRIMARY} ({len(graph_dict['relations'])} triples)")
elif graphs_by_source:
    graph_dict = next(iter(graphs_by_source.values()))
    print(f"⚠️  {KG_SOURCE_PRIMARY} no disponible, usando: {graph_dict['meta']['kg_source']}")
else:
    raise RuntimeError("No se pudo generar ningún grafo.")


[  chunks] 10 elementos (4,329 chars) — generando elemento a elemento...


[  chunks]   1/10  244170-DOB-AVO-05-V00-A0__c0000           +   6 triples  (5.1s)  ETA 0.8 min
[  chunks]   2/10  244170-DOB-AVO-05-V00-A0__c0001           +  10 triples  (5.0s)  ETA 0.7 min
[  chunks]   3/10  244170-DOB-AVO-05-V00-A0__c0002           +   3 triples  (3.3s)  ETA 0.5 min
[  chunks]   4/10  244170-DOB-AVO-05-V00-A0__c0003           +   6 triples  (4.3s)  ETA 0.4 min
[  chunks]   5/10  244170-DOB-AVO-07-V00-A0__c0000           +   8 triples  (4.4s)  ETA 0.4 min
[  chunks]   6/10  244170-DOB-AVO-07-V00-A0__c0001           +  11 triples  (4.8s)  ETA 0.3 min
[  chunks]   7/10  244170-DOB-AVO-07-V00-A0__c0002           +   0 triples  (2.5s)  ETA 0.2 min
[  chunks]   8/10  244170-DOB-AVO-07-V00-A0__c0003           +   6 triples  (28.3s)  ETA 0.2 min
[  chunks]   9/10  244170-DOB-AVO-07-V00-A0__c0004           +   8 triples  (5.5s)  ETA 0.1 min
[  chunks]  10/10  244170-DOB-AVO-09-V00__c0000              +   7 triples  (5.3s)  ETA 0.0 min
[  chunks] Agregando 10 grafos parcial

### 2.1 (Opcional) Subir grafo a GCS

Mismo bucket que el Chroma (`GCP_BUCKET_NAME`), prefijo `graph/`. Salta esta celda si no estás desplegando.


In [7]:
def upload_graph_to_gcs(local_path: Path, bucket_name: str, prefix: str = "graph"):
    if not bucket_name:
        print("⚠️  GCS_BUCKET vacío — exporta GCP_BUCKET_NAME en .env para subir")
        return
    from google.cloud import storage
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob_name = f"{prefix}/{local_path.name}"
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(str(local_path))
    print(f"☁️  Subido: gs://{bucket_name}/{blob_name}")


# Descomenta para subir:
# upload_graph_to_gcs(GRAPH_JSON_PATH, GCS_BUCKET, GCS_GRAPH_PREFIX)


## 3. Retrieval del grafo

Embebemos cada triple (`"sujeto relación objeto"`) con **E5** una sola vez y buscamos por similitud coseno contra la query. Misma representación semántica que usa el dense retriever de chunks.


In [ ]:
def triple_to_text(rel):
    s, r, o = rel
    return f"{s} {r} {o}"


relations = [tuple(r) for r in graph_dict["relations"]]
triple_texts = [triple_to_text(r) for r in relations]

print(f"Embebiendo {len(triple_texts)} triples con E5...")
t0 = time.perf_counter()
triple_vecs = embedder.embed_batch(triple_texts, is_query=False, batch_size=32)
triple_mat = np.array(triple_vecs, dtype=np.float32)
# Normalizar para coseno
triple_norm = triple_mat / (np.linalg.norm(triple_mat, axis=1, keepdims=True) + 1e-12)
print(f"✅ Triples embebidos en {time.perf_counter() - t0:.1f}s — shape={triple_mat.shape}")


Embebiendo 13 triples con E5...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Triples embebidos en 0.1s — shape=(13, 768)


In [ ]:
def retrieve_graph(query, k=GRAPH_TOP_K):
    """Devuelve los k triples más relevantes para la query (cosine con E5)."""
    q_vec = np.array(embedder.embed_one(query, is_query=True), dtype=np.float32)
    q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-12)
    sims = triple_norm @ q_vec
    top_idx = np.argsort(-sims)[:k]
    return [
        {
            "triple": relations[i],
            "text": triple_texts[i],
            "score": float(sims[i]),
            "rank_graph": rank,
        }
        for rank, i in enumerate(top_idx)
    ]


# Sanity check
sample = retrieve_graph("incidencias en el talud", k=5)
for t in sample:
    print(f"  cos={t['score']:.3f} | {t['text']}")


  cos=0.786 | cubierta plana ejecutada en edificio
  cos=0.786 | panel sándwich instalación de cubierta de la marquesina de la vía 02
  cos=0.785 | Trabajos en Cubierta del Edificio solucionando humedades
  cos=0.781 | Trabajos en Cubierta del Edificio incluyen desoxidado
  cos=0.780 | Visita de obra asisten Dirección Facultativa


## 4. Fusión RRF triple: dense + BM25 + graph

El grafo no recupera *chunks*, recupera *hechos*. Lo combinamos de dos formas:

- **Boost de chunks**: para cada triple top-K, localizamos los chunks cuyo texto contiene al sujeto u objeto del triple y aportamos un rank extra a la RRF.
- **Contexto adicional en el prompt**: los triples top-K se pasan al LLM como sección "Hechos relacionados".


In [ ]:
def chunks_matching_triple(triple, docs, metas):
    """Devuelve índices de chunks cuyo texto menciona el sujeto u objeto del triple."""
    s, _, o = triple
    s_low = s.lower()
    o_low = o.lower()
    hits = []
    for i, d in enumerate(docs):
        d_low = d.lower()
        if s_low in d_low or o_low in d_low:
            hits.append(i)
    return hits


def rrf_fusion_graph(dense, bm25, graph_triples, docs, metas, rrf_k=60, top_n=10, min_cos=GRAPH_MIN_COS):
    """Fusión RRF dense + BM25 + grafo.

    El grafo SOLO boostea chunks si el triple supera `min_cos`. Sin umbral, queries
    cuyo tema no está en el grafo recibirían boost de triples mediocres y degradarían
    los resultados (verificado en sub2: hybrid empuja chunks de cubiertas para queries
    sobre talud).
    """
    scores = {}
    for item in dense:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0, "graph_boost": 0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_dense"])
    for item in bm25:
        cid = item["meta"]["chunk_id"]
        scores.setdefault(cid, {"text": item["text"], "meta": item["meta"], "score": 0.0, "graph_boost": 0})
        scores[cid]["score"] += 1.0 / (rrf_k + item["rank_bm25"])
    # Graph boost con umbral
    used = 0
    for t in graph_triples:
        if t["score"] < min_cos:
            continue
        used += 1
        idxs = chunks_matching_triple(t["triple"], docs, metas)
        for i in idxs:
            cid = metas[i]["chunk_id"]
            scores.setdefault(cid, {"text": docs[i], "meta": metas[i], "score": 0.0, "graph_boost": 0})
            scores[cid]["score"] += 1.0 / (rrf_k + t["rank_graph"])
            scores[cid]["graph_boost"] += 1
    # Devuelve también cuántos triples superaron el umbral (para diagnóstico)
    ranked = sorted(scores.values(), key=lambda x: x["score"], reverse=True)[:top_n]
    if ranked and used == 0:
        ranked[0]["_no_graph_boost"] = True
    return ranked


def retrieve_graph_rag(query, top_n=TOP_N):
    d = dense_search(collection, embedder, query, k=RETRIEVAL_K)
    b = bm25_search(query, bm25_index, all_docs, all_metas, stemmer, k=RETRIEVAL_K)
    g = retrieve_graph(query, k=GRAPH_TOP_K)
    return rrf_fusion_graph(d, b, g, all_docs, all_metas, rrf_k=RRF_K, top_n=top_n), g


## 5. Prompt builder enriquecido y `ask_cosora_graph`


In [ ]:
def build_prompt_graph(query, chunks, graph_triples):
    context_blocks = []
    for i, ch in enumerate(chunks, 1):
        doc_id = ch["meta"]["doc_id"]
        text = strip_doc_prefix(ch["text"], EMBED_BACKEND)
        boost = ch.get("graph_boost", 0)
        tag = f" [grafo×{boost}]" if boost else ""
        context_blocks.append(f"[Fragmento {i} - Fuente: {doc_id}{tag}]\n{text}")
    context = "\n\n".join(context_blocks)

    facts = "\n".join(f"- {t['text']}" for t in graph_triples) if graph_triples else "(ninguno)"

    return f"""Eres COSORA, un asistente técnico especializado en análisis de actas de obra ferroviaria en España.

REGLAS ESTRICTAS:
1. Responde ÚNICAMENTE con información del CONTEXTO o de los HECHOS RELACIONADOS.
2. Si la información no está, dilo explícitamente.
3. Cita siempre la fuente entre paréntesis: (Fuente: nombre_del_acta).
4. Los HECHOS RELACIONADOS son triples extraídos automáticamente; úsalos como pistas, no como fuente primaria.

VOCABULARIO: DF=Dirección Facultativa, UTE=Unión Temporal de Empresas, DEO=Director de Ejecución de Obra, DO=Director de Obra.

=== HECHOS RELACIONADOS (grafo) ===
{facts}

=== CONTEXTO (fragmentos) ===
{context}

=== PREGUNTA ===
{query}

=== RESPUESTA ==="""


def generate_answer(prompt, model="gpt-4o-mini"):
    from openai import OpenAI
    api_key = os.getenv("OPENAI_API_KEY")
    client = OpenAI(api_key=api_key)
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return resp.choices[0].message.content


def filter_chunks(chunks):
    if not chunks or chunks[0]["score"] < RRF_MIN_SCORE:
        return None
    max_score = chunks[0]["score"]
    return [c for c in chunks if c["score"] >= max_score * SCORE_RATIO]


def ask_cosora_graph(query, verbose=True):
    chunks, triples = retrieve_graph_rag(query, top_n=TOP_N)
    filtered = filter_chunks(chunks)
    if filtered is None:
        return "No se encontró información suficientemente relevante en las actas."
    prompt = build_prompt_graph(query, filtered, triples)
    answer = generate_answer(prompt)
    if verbose:
        print(f"Query: {query}")
        print(f"\nTriples top-{len(triples)}:")
        for t in triples:
            print(f"  cos={t['score']:.3f} | {t['text']}")
        print(f"\nChunks ({len(filtered)}):")
        for c in filtered:
            boost = f" g×{c.get('graph_boost', 0)}" if c.get('graph_boost') else ""
            print(f"  - [{c['meta']['doc_id']}] score={c['score']:.4f}{boost}")
        print(f"\nRespuesta:\n{answer}\n" + "-"*70)
    return answer


# Test rápido
_ = ask_cosora_graph("¿Qué se decidió sobre el talud?", verbose=True)


Query: ¿Qué se decidió sobre el talud?

Triples top-5:
  cos=0.741 | panel sándwich instalación de cubierta de la marquesina de la vía 02
  cos=0.738 | Trabajos en Cubierta del Edificio incluyen desoxidado
  cos=0.737 | Trabajos en Cubierta del Edificio solucionando humedades
  cos=0.733 | cubierta plana ejecutada en edificio
  cos=0.733 | zanja de unión realización de arqueta

Chunks (8):
  - [244170-DOB-AVO-05-V00-A0] score=0.0633 g×3
  - [244170-DOB-AVO-07-V00-A0] score=0.0617 g×3
  - [244170-DOB-AVO-05-V00-A0] score=0.0466 g×2
  - [244170-DOB-AVO-07-V00-A0] score=0.0462 g×2
  - [244170-DOB-AVO-07-V00-A0] score=0.0448 g×2
  - [244170-DOB-AVO-16-V00-A03] score=0.0415 g×2
  - [244170-DOB-AVO-17-V00-A0] score=0.0410 g×2
  - [244170-DOB-AVO-16-V00-A03] score=0.0409 g×2

Respuesta:
La información sobre el talud no está disponible en el contexto proporcionado. (Fuente: no disponible)
----------------------------------------------------------------------


## 6. Evaluación LLM-as-Judge — baseline vs Graph RAG

Mismo juez (`gpt-4o-mini`) y mismas queries que en `query_pipeline.ipynb`. Comparamos:

- **baseline**: dense (E5) + BM25 + RRF
- **graph**: dense + BM25 + grafo + RRF triple

Métricas: Precision@K, MRR, NDCG@K, MAP.

> Coste: ~0,02 USD para 8 queries × 2 variantes × 5 chunks.


In [ ]:
import pandas as pd
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)
LLM_JUDGE_MODEL = "gpt-4o-mini"

BENCHMARK_QUERIES_FACTUAL = [
    "¿Qué se decidió sobre el talud?",
    "¿Cuál es el estado del camino provisional?",
    "¿Qué incidencias AR-29 aparecen?",
    "¿Qué responsable está asignado a las acciones sobre el talud?",
    "¿Qué se acordó sobre hormigonado de zapatas?",
    "Estado de las instalaciones de megafonía",
    "¿Qué documentación debe aportar la UTE sobre estabilidad?",
]
# Queries transversales/analíticas — extraídas de docs/queries.md (BLOQUES 1, 2, 4, 5, 10)
# Es donde se espera que el grafo aporte valor frente al baseline.
BENCHMARK_QUERIES_TRANSVERSAL = [
    "¿Cuáles son las incidencias más frecuentes en todas las actas?",
    "¿Qué elementos constructivos presentan más incidencias?",
    "¿Qué problemas pueden afectar a la seguridad de la explotación ferroviaria?",
    "Agrupa todas las incidencias detectadas en las actas y clasifícalas por tipo",
    "¿Cuáles son las acciones pendientes más frecuentes en las actas?",
    "Compara las diferentes obras en función del número de incidencias",
]
BENCHMARK_QUERIES = BENCHMARK_QUERIES_FACTUAL + BENCHMARK_QUERIES_TRANSVERSAL
EVAL_K = 5


JUDGE_DEBUG = False  # Pon True para imprimir respuestas crudas de las primeras 3 llamadas
_judge_debug_remaining = 3


def judge_relevance(query: str, chunk_text: str) -> int:
    global _judge_debug_remaining
    system = (
        "Eres un evaluador de relevancia para un sistema de búsqueda sobre actas de obra "
        "ferroviaria. Tu única tarea: emitir un dígito (0, 1 o 2) que valore si el FRAGMENTO "
        "es relevante para la PREGUNTA. Sin texto adicional."
    )
    user = (
        f"PREGUNTA: {query}\n\n"
        f"FRAGMENTO:\n{chunk_text[:600]}\n\n"
        "Criterios:\n"
        "0 = el fragmento no aporta nada a la pregunta\n"
        "1 = menciona el tema pero no contesta\n"
        "2 = contiene información que responde directa o sustancialmente a la pregunta\n\n"
        "Responde SOLO con el dígito 0, 1 o 2."
    )
    try:
        resp = client.chat.completions.create(
            model=LLM_JUDGE_MODEL,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            temperature=0,
            max_tokens=2,
        )
        raw = resp.choices[0].message.content or ""
    except Exception as e:
        if JUDGE_DEBUG:
            print(f"  judge_relevance EXC: {type(e).__name__}: {e}")
        return -1  # marcador de error, no se cuenta como irrelevante
    ans = raw.strip()
    if JUDGE_DEBUG and _judge_debug_remaining > 0:
        print(f"  judge raw={raw!r}  → ans={ans!r}")
        _judge_debug_remaining -= 1
    # Toma el PRIMER dígito 0/1/2 del string (la respuesta debería empezar por él)
    for ch in ans:
        if ch in "012":
            return int(ch)
    return -1  # respuesta no parseable, no se cuenta


def dcg_at_k(rels, k):
    return sum(r / np.log2(i + 2) for i, r in enumerate(rels[:k]))

def ndcg_at_k(rels, k):
    dcg = dcg_at_k(rels, k)
    ideal = dcg_at_k(sorted(rels, reverse=True), k)
    return dcg / ideal if ideal > 0 else 0.0

def average_precision(rels):
    rel_count, psum = 0, 0.0
    for i, r in enumerate(rels):
        if r >= 1:
            rel_count += 1
            psum += rel_count / (i + 1)
    return psum / rel_count if rel_count else 0.0

def mrr(rels):
    for i, r in enumerate(rels):
        if r >= 1:
            return 1.0 / (i + 1)
    return 0.0


def _clean_rels(rels):
    """Sustituye -1 (error de juez) por 0 para que las métricas no se rompan, pero las contamos."""
    return [max(r, 0) for r in rels]


def eval_variant(name, retrieve_fn, queries, k=EVAL_K):
    judgments = []
    n_errors = 0
    for q in queries:
        hits = retrieve_fn(q, k)
        rels = []
        for h in hits:
            txt = strip_doc_prefix(h["text"], EMBED_BACKEND)
            r = judge_relevance(q, txt)
            if r == -1:
                n_errors += 1
            rels.append(r)
        clean = _clean_rels(rels)
        judgments.append({
            "query": q,
            "relevances": rels,
            "clean_relevances": clean,
            f"precision@{k}": sum(1 for r in clean if r >= 1) / k,
            "mrr": mrr(clean),
            f"ndcg@{k}": ndcg_at_k(clean, k),
            "ap": average_precision(clean),
        })
    if n_errors:
        print(f"  ⚠️  {name}: {n_errors} juicios no parseables (ver JUDGE_DEBUG=True)")
    return {
        "variant": name,
        f"Precision@{k}": np.mean([j[f"precision@{k}"] for j in judgments]),
        "MRR": np.mean([j["mrr"] for j in judgments]),
        f"NDCG@{k}": np.mean([j[f"ndcg@{k}"] for j in judgments]),
        "MAP": np.mean([j["ap"] for j in judgments]),
    }, judgments


def retrieve_baseline_k(q, k):
    return retrieve_baseline(q, top_n=k)

def retrieve_graph_k(q, k):
    chunks, _ = retrieve_graph_rag(q, top_n=k)
    return chunks


print("Evaluando baseline...")
base_summary, base_detail = eval_variant("baseline", retrieve_baseline_k, BENCHMARK_QUERIES)
print("Evaluando graph...")
graph_summary, graph_detail = eval_variant("graph", retrieve_graph_k, BENCHMARK_QUERIES)

df = pd.DataFrame([base_summary, graph_summary]).set_index("variant")
print("\n🏆 Resultados:")
display(df)

# Delta por query
delta_rows = []
for i, q in enumerate(BENCHMARK_QUERIES):
    delta_rows.append({
        "query": q[:50],
        "base_P@K": base_detail[i][f"precision@{EVAL_K}"],
        "graph_P@K": graph_detail[i][f"precision@{EVAL_K}"],
        "base_NDCG": base_detail[i][f"ndcg@{EVAL_K}"],
        "graph_NDCG": graph_detail[i][f"ndcg@{EVAL_K}"],
    })
df_delta = pd.DataFrame(delta_rows)
df_delta["Δ NDCG"] = df_delta["graph_NDCG"] - df_delta["base_NDCG"]
display(df_delta.sort_values("Δ NDCG", ascending=False))


Evaluando baseline...
Evaluando graph...

🏆 Resultados:


,Precision@5,MRR,NDCG@5,MAP
variant,,,,
baseline,0.825,0.916667,0.923969,0.916667
graph,0.375,0.447917,0.532426,0.459896


,query,base_P@K,graph_P@K,base_NDCG,graph_NDCG,Δ NDCG
3,¿Cuáles son las incidencias más frecuentes en ...,0.2,0.4,0.500000,0.919721,0.419721
5,¿Qué se acordó sobre hormigonado de zapatas?,1.0,1.0,1.000000,1.000000,0.000000
7,¿Qué documentación debe aportar la UTE sobre e...,1.0,0.8,1.000000,0.760640,-0.239360
4,¿Qué responsable está asignado a las acciones ...,1.0,0.2,1.000000,0.630930,-0.369070
1,¿Cuál es el estado del camino provisional?,1.0,0.4,0.906528,0.517442,-0.389086
6,Estado de las instalaciones de megafonía,0.8,0.2,0.985227,0.430677,-0.554550
2,¿Qué incidencias AR-29 aparecen?,0.6,0.0,1.000000,0.000000,-1.000000
0,¿Qué se decidió sobre el talud?,1.0,0.0,1.000000,0.000000,-1.000000


## 7. Pruebas interactivas


In [ ]:
MIS_PREGUNTAS = [
    "¿Qué se decidió sobre el talud?",
    "¿Cuál es el estado del camino provisional?",
    "¿Qué incidencias aparecen relacionadas con instalaciones eléctricas?",
]

for q in MIS_PREGUNTAS:
    print("="*80)
    print(f"👤 {q}")
    print("-"*80)
    resp = ask_cosora_graph(q, verbose=False)
    print(f"🤖 {resp}")


👤 ¿Qué se decidió sobre el talud?
--------------------------------------------------------------------------------
🤖 La información sobre el talud no está disponible en el contexto proporcionado. (Fuente: no disponible)
👤 ¿Cuál es el estado del camino provisional?
--------------------------------------------------------------------------------
🤖 Durante la visita de obra, la Dirección Facultativa constató que se está continuando con el rebaje del nivel de tierras del camino provisional, lo que supone una mejora de las condiciones de seguridad y estabilidad del entorno, ya que disminuye el empuje de tierras hacia las parcelas. Sin embargo, se observó la presencia de tierras desprendidas y acumuladas en contacto directo con los muros de los patios colindantes de las viviendas (Fuente: 254275-DO-AVO-19-V07).
👤 ¿Qué incidencias aparecen relacionadas con instalaciones eléctricas?
--------------------------------------------------------------------------------
🤖 Las incidencias relacionadas 

## 8. Comparativa global — baseline vs grafo (chunks / hybrid / original)

Evalúa **baseline + los 3 grafos que generó la sección 2** sobre las mismas queries de benchmark. Si alguno falló, se omite y se sigue.

> Coste: ~0,02 USD por cada variante de grafo presente (8 queries × 5 chunks × `gpt-4o-mini`).


In [ ]:
def prepare_variant(gd):
    """Embebe los triples de un grafo (dict) para usarlos en retrieval."""
    rels = [tuple(r) for r in gd["relations"]]
    texts = [f"{s} {r} {o}" for s, r, o in rels]
    if not texts:
        return None
    vecs = embedder.embed_batch(texts, is_query=False, batch_size=32)
    mat = np.array(vecs, dtype=np.float32)
    norm = mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)
    return {"meta": gd.get("meta", {}), "relations": rels, "texts": texts, "norm": norm}


def retrieve_graph_variant(query, variant, k=GRAPH_TOP_K):
    q = np.array(embedder.embed_one(query, is_query=True), dtype=np.float32)
    q = q / (np.linalg.norm(q) + 1e-12)
    sims = variant["norm"] @ q
    top = np.argsort(-sims)[:k]
    return [
        {"triple": variant["relations"][i], "text": variant["texts"][i], "score": float(sims[i]), "rank_graph": rk}
        for rk, i in enumerate(top)
    ]


def retrieve_graph_rag_with(query, variant, top_n=TOP_N):
    d = dense_search(collection, embedder, query, k=RETRIEVAL_K)
    b = bm25_search(query, bm25_index, all_docs, all_metas, stemmer, k=RETRIEVAL_K)
    g = retrieve_graph_variant(query, variant, k=GRAPH_TOP_K)
    return rrf_fusion_graph(d, b, g, all_docs, all_metas, rrf_k=RRF_K, top_n=top_n)


loaded_variants = {}
for src, gd in graphs_by_source.items():
    v = prepare_variant(gd)
    if v is not None:
        loaded_variants[src] = v
        print(f"[{src:>8}] {len(v['relations'])} triples embebidos")

print(f"\nVariantes a comparar: {list(loaded_variants)}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[  chunks] 9 triples embebidos


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[  hybrid] 13 triples embebidos


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

[original] 130 triples embebidos

Variantes a comparar: ['chunks', 'hybrid', 'original']


In [ ]:
all_summaries = []
all_details = {}

print("Evaluando baseline (dense + BM25)...")
base_s, base_d = eval_variant("baseline", retrieve_baseline_k, BENCHMARK_QUERIES)
all_summaries.append(base_s)
all_details["baseline"] = base_d

for src, variant in loaded_variants.items():
    name = f"graph_{src}"
    print(f"Evaluando {name}...")
    fn = lambda q, k, _v=variant: retrieve_graph_rag_with(q, _v, top_n=k)
    s, d = eval_variant(name, fn, BENCHMARK_QUERIES)
    all_summaries.append(s)
    all_details[name] = d

df_cmp = pd.DataFrame(all_summaries).set_index("variant")
print("\n🏆 Comparativa global:")
display(df_cmp)

# Delta NDCG por query para cada variante de grafo
delta_cols = {"query": [q[:50] for q in BENCHMARK_QUERIES]}
delta_cols["base_NDCG"] = [d[f"ndcg@{EVAL_K}"] for d in all_details["baseline"]]
for src in loaded_variants:
    delta_cols[f"graph_{src}_NDCG"] = [d[f"ndcg@{EVAL_K}"] for d in all_details[f"graph_{src}"]]
    delta_cols[f"Δ_{src}"] = [
        all_details[f"graph_{src}"][i][f"ndcg@{EVAL_K}"] - all_details["baseline"][i][f"ndcg@{EVAL_K}"]
        for i in range(len(BENCHMARK_QUERIES))
    ]
df_delta = pd.DataFrame(delta_cols)
print("\nΔ NDCG por query (graph - baseline):")
display(df_delta)

# Ganador por métrica
print("\n📊 Ganador por métrica (global):")
for col in [c for c in df_cmp.columns]:
    winner = df_cmp[col].idxmax()
    print(f"  {col:>15} → {winner} ({df_cmp.loc[winner, col]:.3f})")


# Desglose factual vs transversal — donde el grafo se espera que GANE
def _summary_subset(name, details, queries_subset, k=EVAL_K):
    idxs = [i for i, q in enumerate(BENCHMARK_QUERIES) if q in queries_subset]
    if not idxs:
        return None
    return {
        "variant": name,
        f"Precision@{k}": np.mean([details[i][f"precision@{k}"] for i in idxs]),
        "MRR": np.mean([details[i]["mrr"] for i in idxs]),
        f"NDCG@{k}": np.mean([details[i][f"ndcg@{k}"] for i in idxs]),
        "MAP": np.mean([details[i]["ap"] for i in idxs]),
    }


print("\n🎯 Subconjunto FACTUAL (lookup específico):")
rows = [_summary_subset(n, all_details[n], BENCHMARK_QUERIES_FACTUAL) for n in all_details]
display(pd.DataFrame([r for r in rows if r]).set_index("variant"))

print("\n🌐 Subconjunto TRANSVERSAL (agregadas/analíticas — donde el grafo debe ayudar):")
rows = [_summary_subset(n, all_details[n], BENCHMARK_QUERIES_TRANSVERSAL) for n in all_details]
display(pd.DataFrame([r for r in rows if r]).set_index("variant"))


Evaluando baseline (dense + BM25)...
Evaluando graph_chunks...
Evaluando graph_hybrid...
Evaluando graph_original...

🏆 Comparativa global:


,Precision@5,MRR,NDCG@5,MAP
variant,,,,
baseline,0.825,0.916667,0.923969,0.916667
graph_chunks,0.200,0.166667,0.274073,0.198437
graph_hybrid,0.375,0.447917,0.532426,0.459896
graph_original,0.200,0.202083,0.349155,0.215625



Δ NDCG por query (graph - baseline):


,query,base_NDCG,graph_chunks_NDCG,Δ_chunks,graph_hybrid_NDCG,Δ_hybrid,graph_original_NDCG,Δ_original
0,¿Qué se decidió sobre el talud?,1.000000,0.430677,-0.569323,0.000000,-1.000000,0.430677,-0.569323
1,¿Cuál es el estado del camino provisional?,0.906528,0.000000,-0.906528,0.517442,-0.389086,0.386853,-0.519675
2,¿Qué incidencias AR-29 aparecen?,1.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,-1.000000
3,¿Cuáles son las incidencias más frecuentes en ...,0.500000,0.000000,-0.500000,0.919721,0.419721,0.500000,0.000000
4,¿Qué responsable está asignado a las acciones ...,1.000000,0.500000,-0.500000,0.630930,-0.369070,0.000000,-1.000000
5,¿Qué se acordó sobre hormigonado de zapatas?,1.000000,0.501266,-0.498734,1.000000,0.000000,0.501266,-0.498734
6,Estado de las instalaciones de megafonía,0.985227,0.000000,-0.985227,0.430677,-0.554550,0.430677,-0.554550
7,¿Qué documentación debe aportar la UTE sobre e...,1.000000,0.760640,-0.239360,0.760640,-0.239360,0.543771,-0.456229



📊 Ganador por métrica:
      Precision@5 → baseline (0.825)
              MRR → baseline (0.917)
           NDCG@5 → baseline (0.924)
              MAP → baseline (0.917)
